# Projeto – Fase 1: Países e PIB (1960–2021)

Este notebook implementa a coleta e consolidação pedidas no projeto:

1. consome a **API REST** fornecida em `api_fase01.py` para obter os dados dos países;
2. lê `gdp_data.csv` com **pandas**;
3. usa o código do país como chave comum entre as duas fontes;
4. mantém **todos os países retornados pela API**, mesmo quando não existe PIB para algum ou todos os anos;
5. gera o arquivo principal `paises_pib_1960_2021.csv` com as colunas pedidas no enunciado;
6. gera também, de forma opcional, `paises_pib_1960_2021_com_moeda.csv`, preservando `codigo_moeda`, já que esse atributo também é solicitado na etapa de coleta da API.

> **Antes de executar o notebook:** deixe `api_fase01.py`, `country.json`, `gdp_data.csv` e `requirements.txt` na mesma pasta deste notebook.

## 1. Preparação do ambiente e inicialização da API

No terminal, na pasta do projeto, instale as dependências e inicie a API:

```bash
pip install -r requirements.txt
python api_fase01.py
```

A API ficará disponível localmente. O endpoint usado neste notebook é `http://127.0.0.1:5000/country`.

Mantenha o terminal da API aberto enquanto executa as células de coleta abaixo.

In [1]:
from pathlib import Path
import requests
import pandas as pd

BASE_DIR = Path.cwd()
API_URL = "http://127.0.0.1:5000/country"
ARQUIVO_GDP = BASE_DIR / "gdp_data.csv"

ANOS = [str(ano) for ano in range(1960, 2022)]

print(f"Pasta de trabalho: {BASE_DIR}")
print(f"Quantidade de anos esperada: {len(ANOS)}")

Pasta de trabalho: C:\Users\italo\Documents\Projetos PUC RS\Coleta e Preparação de Dados\Fase1\Coleta e Preparação de Dados - Fase 1(Entrega)
Quantidade de anos esperada: 62


## 2. Coleta dos países pela API REST

O endpoint `/country` devolve um objeto JSON em que cada chave é o código de um país. A coleta abaixo usa `requests.get`, valida a resposta HTTP e converte o JSON em um DataFrame.

In [2]:
try:
    resposta = requests.get(API_URL, timeout=30)
    resposta.raise_for_status()
except requests.RequestException as exc:
    raise RuntimeError(
        "Não foi possível acessar a API. Verifique se `python api_fase01.py` "
        "está em execução em outro terminal."
    ) from exc

paises_json = resposta.json()
print(f"Países/territórios retornados pela API: {len(paises_json)}")

Países/territórios retornados pela API: 250


In [3]:
registros_paises = []

for codigo, dados in paises_json.items():
    registros_paises.append({
        "codigo": codigo,
        "nome": dados.get("name"),
        "codigo_moeda": dados.get("currency"),
        "populacao": dados.get("population"),
        "capital": dados.get("capital"),
        "area": dados.get("area"),
        "continente": dados.get("continent"),
    })

df_paises_api = (
    pd.DataFrame(registros_paises)
      .sort_values("codigo")
      .reset_index(drop=True)
)

df_paises_api.head()

,codigo,nome,codigo_moeda,populacao,capital,area,continente
0,ABW,Aruba,AWG,105845,Oranjestad,193.0,North America
1,AFG,Afghanistan,AFN,37172386,Kabul,647500.0,Asia
2,AGO,Angola,AOA,30809762,Luanda,1246700.0,Africa
3,AIA,Anguilla,XCD,13254,The Valley,102.0,North America
4,ALA,Aland,EUR,26711,Mariehamn,1580.0,Europe


In [4]:
df_paises_api.info()

<class 'pandas.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   codigo        250 non-null    str    
 1   nome          250 non-null    str    
 2   codigo_moeda  250 non-null    str    
 3   populacao     250 non-null    int64  
 4   capital       250 non-null    str    
 5   area          250 non-null    float64
 6   continente    250 non-null    str    
dtypes: float64(1), int64(1), str(5)
memory usage: 13.8 KB


## 3. Leitura dos dados de PIB

O arquivo `gdp_data.csv` contém, além das colunas de identificação, uma coluna para cada ano de 1960 a 2021. Algumas linhas representam regiões/agregados econômicos e não países da API; elas serão eliminadas pelo alinhamento usando `codigo`.

In [5]:
df_gdp_raw = pd.read_csv(ARQUIVO_GDP)

# Remove colunas totalmente vazias (por exemplo, uma coluna final sem nome).
df_gdp_raw = df_gdp_raw.dropna(axis=1, how="all")

colunas_ausentes = [ano for ano in ANOS if ano not in df_gdp_raw.columns]
if colunas_ausentes:
    raise ValueError(f"Anos ausentes no arquivo de PIB: {colunas_ausentes}")

print(f"Linhas no arquivo de PIB: {len(df_gdp_raw)}")
print(f"Primeiro ano: {ANOS[0]} | Último ano: {ANOS[-1]}")
df_gdp_raw.head()

Linhas no arquivo de PIB: 266
Primeiro ano: 1960 | Último ano: 2021


,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
0,Aruba,ABW,GDP (current US$),NY.GDP.MKTP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,2.615084e+09,2.727933e+09,2.791061e+09,2.963128e+09,2.983799e+09,3.092179e+09,3.202235e+09,3.368970e+09,2.610039e+09,3.126019e+09
1,Africa Eastern and Southern,AFE,GDP (current US$),NY.GDP.MKTP.CD,2.129081e+10,2.180870e+10,2.370727e+10,2.821034e+10,2.611906e+10,2.968249e+10,...,9.725734e+11,9.834729e+11,1.003768e+12,9.245228e+11,8.827213e+11,1.021119e+12,1.007240e+12,1.001017e+12,9.274845e+11,1.080712e+12
2,Afghanistan,AFG,GDP (current US$),NY.GDP.MKTP.CD,5.377778e+08,5.488889e+08,5.466667e+08,7.511112e+08,8.000000e+08,1.006667e+09,...,2.020357e+10,2.056449e+10,2.055058e+10,1.999816e+10,1.801956e+10,1.889635e+10,1.841885e+10,1.890449e+10,2.014344e+10,1.478686e+10
3,Africa Western and Central,AFW,GDP (current US$),NY.GDP.MKTP.CD,1.040414e+10,1.112789e+10,1.194319e+10,1.267633e+10,1.383837e+10,1.486223e+10,...,7.360399e+11,8.322169e+11,8.924979e+11,7.669580e+11,6.905454e+11,6.837480e+11,7.663597e+11,7.947191e+11,7.847997e+11,8.401873e+11
4,Angola,AGO,GDP (current US$),NY.GDP.MKTP.CD,NaN,NaN,NaN,NaN,NaN,NaN,...,1.249982e+11,1.334016e+11,1.372444e+11,8.721930e+10,4.984049e+10,6.897277e+10,7.779294e+10,6.930911e+10,5.361907e+10,6.740429e+10


In [6]:
df_pib = (
    df_gdp_raw[["Country Code"] + ANOS]
      .rename(columns={"Country Code": "codigo"})
      .copy()
)

# Garante uma linha por código no arquivo de PIB.
if df_pib["codigo"].duplicated().any():
    duplicados = df_pib.loc[df_pib["codigo"].duplicated(), "codigo"].tolist()
    raise ValueError(f"Códigos duplicados no arquivo de PIB: {duplicados}")

print(f"Códigos únicos no arquivo de PIB: {df_pib['codigo'].nunique()}")

Códigos únicos no arquivo de PIB: 266


## 4. Consolidação das duas fontes

A chave comum é o **código do país** (`codigo`). Para seguir a dica do README, a consolidação abaixo usa `pd.concat` após alinhar o DataFrame de PIB exatamente à ordem dos códigos retornados pela API. Assim, todos os países da API permanecem no resultado e códigos extras existentes somente no arquivo de PIB não entram no conjunto final.

In [7]:
paises_indexados = df_paises_api.set_index("codigo")

# Reindexar é o ponto que garante que a referência do conjunto final seja a API.
pib_indexado = (
    df_pib.set_index("codigo")
          .reindex(paises_indexados.index)
)

df_consolidado = (
    pd.concat([paises_indexados, pib_indexado], axis=1)
      .reset_index()
)

df_consolidado.head()

,codigo,nome,codigo_moeda,populacao,capital,area,continente,1960,1961,1962,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
0,ABW,Aruba,AWG,105845,Oranjestad,193.0,North America,NaN,NaN,NaN,...,2.615084e+09,2.727933e+09,2.791061e+09,2.963128e+09,2.983799e+09,3.092179e+09,3.202235e+09,3.368970e+09,2.610039e+09,3.126019e+09
1,AFG,Afghanistan,AFN,37172386,Kabul,647500.0,Asia,5.377778e+08,5.488889e+08,5.466667e+08,...,2.020357e+10,2.056449e+10,2.055058e+10,1.999816e+10,1.801956e+10,1.889635e+10,1.841885e+10,1.890449e+10,2.014344e+10,1.478686e+10
2,AGO,Angola,AOA,30809762,Luanda,1246700.0,Africa,NaN,NaN,NaN,...,1.249982e+11,1.334016e+11,1.372444e+11,8.721930e+10,4.984049e+10,6.897277e+10,7.779294e+10,6.930911e+10,5.361907e+10,6.740429e+10
3,AIA,Anguilla,XCD,13254,The Valley,102.0,North America,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ALA,Aland,EUR,26711,Mariehamn,1580.0,Europe,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. DataFrame final e validações

O enunciado final lista como colunas de saída: `codigo`, `nome`, `populacao`, `capital`, `area`, `continente` e uma coluna de PIB para cada ano de 1960 a 2021. Por isso, `codigo_moeda` é coletado da API e mantido no DataFrame consolidado, mas não entra no CSV principal. Uma versão estendida com moeda também será exportada.

In [8]:
COLUNAS_BASE = ["codigo", "nome", "populacao", "capital", "area", "continente"]
COLUNAS_FINAIS = COLUNAS_BASE + ANOS

# CSV principal: exatamente as colunas explicitadas na especificação final.
df_final = df_consolidado[COLUNAS_FINAIS].copy()

# CSV estendido: atende também à solicitação de preservar o código da moeda.
COLUNAS_ESTENDIDAS = [
    "codigo", "nome", "codigo_moeda", "populacao", "capital", "area", "continente"
] + ANOS
df_final_com_moeda = df_consolidado[COLUNAS_ESTENDIDAS].copy()

assert len(df_final) == len(df_paises_api), "O número de linhas deve ser igual ao número de países da API."
assert df_final["codigo"].is_unique, "Cada país deve ocupar uma única linha."
assert ANOS[0] == "1960" and ANOS[-1] == "2021" and len(ANOS) == 62
assert list(df_final.columns) == COLUNAS_FINAIS

print(f"Linhas finais: {len(df_final)}")
print(f"Colunas finais: {len(df_final.columns)}")
print(f"Países sem nenhum PIB disponível: {df_final[ANOS].isna().all(axis=1).sum()}")
print(f"Valores de PIB ausentes no painel: {int(df_final[ANOS].isna().sum().sum())}")

Linhas finais: 250
Colunas finais: 68
Países sem nenhum PIB disponível: 37
Valores de PIB ausentes no painel: 5143


In [9]:
df_final.head()

,codigo,nome,populacao,capital,area,continente,1960,1961,1962,1963,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
0,ABW,Aruba,105845,Oranjestad,193.0,North America,NaN,NaN,NaN,NaN,...,2.615084e+09,2.727933e+09,2.791061e+09,2.963128e+09,2.983799e+09,3.092179e+09,3.202235e+09,3.368970e+09,2.610039e+09,3.126019e+09
1,AFG,Afghanistan,37172386,Kabul,647500.0,Asia,5.377778e+08,5.488889e+08,5.466667e+08,7.511112e+08,...,2.020357e+10,2.056449e+10,2.055058e+10,1.999816e+10,1.801956e+10,1.889635e+10,1.841885e+10,1.890449e+10,2.014344e+10,1.478686e+10
2,AGO,Angola,30809762,Luanda,1246700.0,Africa,NaN,NaN,NaN,NaN,...,1.249982e+11,1.334016e+11,1.372444e+11,8.721930e+10,4.984049e+10,6.897277e+10,7.779294e+10,6.930911e+10,5.361907e+10,6.740429e+10
3,AIA,Anguilla,13254,The Valley,102.0,North America,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ALA,Aland,26711,Mariehamn,1580.0,Europe,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 6. Exportação dos arquivos CSV

Os valores ausentes de PIB são exportados como campos vazios, pois a ausência de informação para alguns países/anos é permitida pelo projeto.

In [10]:
SAIDA_PRINCIPAL = BASE_DIR / "paises_pib_1960_2021.csv"
SAIDA_COM_MOEDA = BASE_DIR / "paises_pib_1960_2021_com_moeda.csv"

df_final.to_csv(SAIDA_PRINCIPAL, index=False, encoding="utf-8-sig", na_rep="")
df_final_com_moeda.to_csv(SAIDA_COM_MOEDA, index=False, encoding="utf-8-sig", na_rep="")

print(f"Arquivo principal gerado: {SAIDA_PRINCIPAL}")
print(f"Arquivo estendido gerado: {SAIDA_COM_MOEDA}")

Arquivo principal gerado: C:\Users\italo\Documents\Projetos PUC RS\Coleta e Preparação de Dados\Fase1\Coleta e Preparação de Dados - Fase 1(Entrega)\paises_pib_1960_2021.csv
Arquivo estendido gerado: C:\Users\italo\Documents\Projetos PUC RS\Coleta e Preparação de Dados\Fase1\Coleta e Preparação de Dados - Fase 1(Entrega)\paises_pib_1960_2021_com_moeda.csv


## 7. Checagem rápida do resultado

Esta última célula relê o CSV principal e confirma que o arquivo exportado continua com uma linha por código da API e com os anos de 1960 a 2021.

In [11]:
df_teste = pd.read_csv(SAIDA_PRINCIPAL)

assert len(df_teste) == len(df_paises_api)
assert df_teste["codigo"].nunique() == len(df_paises_api)
assert all(ano in df_teste.columns for ano in ANOS)

print("Validação concluída com sucesso.")
print(df_teste.shape)
df_teste.head(3)

Validação concluída com sucesso.
(250, 68)


,codigo,nome,populacao,capital,area,continente,1960,1961,1962,1963,...,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021
0,ABW,Aruba,105845,Oranjestad,193.0,North America,NaN,NaN,NaN,NaN,...,2.615084e+09,2.727933e+09,2.791061e+09,2.963128e+09,2.983799e+09,3.092179e+09,3.202235e+09,3.368970e+09,2.610039e+09,3.126019e+09
1,AFG,Afghanistan,37172386,Kabul,647500.0,Asia,5.377778e+08,5.488889e+08,5.466667e+08,7.511112e+08,...,2.020357e+10,2.056449e+10,2.055058e+10,1.999816e+10,1.801956e+10,1.889635e+10,1.841885e+10,1.890449e+10,2.014344e+10,1.478686e+10
2,AGO,Angola,30809762,Luanda,1246700.0,Africa,NaN,NaN,NaN,NaN,...,1.249982e+11,1.334016e+11,1.372444e+11,8.721930e+10,4.984049e+10,6.897277e+10,7.779294e+10,6.930911e+10,5.361907e+10,6.740429e+10
